# Reliability Ops
# 0. 介绍

**研究背景**：Agent 完成一项任务时，会连续调用大模型和工具，并经过解析、重试和状态更新等多个步骤。一个早期错误可能继续传到后面，变成重复调用、下游异常或最终任务失败，因此外层程序不仅要记录“发生了什么”，还要能回答“最早错在哪一步、属于哪个模块、是否正在循环”。

**现存问题**：工业界和生产系统中真实出现过的错误基线，是只保存平铺的文本日志或最后一条异常，失败后再盲目重试。同一个工具因参数或返回结构错误而被重复调用时，后续的二次错误和大量告警会淹没最初根因；运维人员最后只看到“任务失败”，无法区分是模型决策、工具协议还是 Harness 生命周期出错，也无法稳定复现和回归检查。

**解决方案**：本 Notebook 将实现一个极简的 Reliability Ops，采用`结构化 Trace + 因果错误传播 + 失败分类 + 循环检测 + Run Summary`机制：按 OpenTelemetry 及 GenAI 语义约定为每个步骤记录 Span、父子关系、异常字段和停止原因，用明确的因果链保留错误如何向下游传播；再按 Harness 模块分类失败，用“工具名 + 规范化参数”签名识别重复调用，最后从终态反向找到最早根因并生成可机器读取的运行摘要。然后在同一份真实 API 运行轨迹上进行对比：基线版本只保留末次错误，无法定位根因和重复调用；改进版本从同一条 trace 中恢复错误传播链、识别循环并输出稳定诊断，从而直观看到可靠性运维的核心不是“多打几行日志”，而是让每个失败都能被分类、关联、追溯和验证。

## 目录

0. 介绍
1. 初始化真实 API
2. 前置准备
3. 获取并验证 API 响应
4. 定义基线组件 *
5. 展示基线故障 *
6. 定义改进组件 *
7. 展示修复结果 *
8. 汇总消融对照

# 1. 初始化真实 API
## 连接大模型
程序需要先读取项目 `.env` 文件中已经准备好的连接信息，才能使用真实的大模型。本节直接读取这些信息并建立连接，同时保存后面要使用的模型名称。

In [1]:
from dotenv import dotenv_values, find_dotenv
from openai import OpenAI

config = dotenv_values(find_dotenv())  # 自动找到并读取项目的 .env
client = OpenAI(
    api_key=config["OPENAI_API_KEY"],
    base_url=config["OPENAI_BASE_URL"],
)
model_name = config["OPENAI_MODEL"]
print(f"真实 API 已就绪：{model_name}")

真实 API 已就绪：LongCat-2.0


输出显示了模型名称，说明真实 API 已经准备好，但此时还没有向大模型发送请求。下一章将定义大模型可以使用的工具，以及需要完成的多步任务。

# 2. 前置准备
## 2.1 固定模型的工具格式
Reliability Ops 需要知道每一步究竟调用了什么，才能把模型决策、工具执行和后续错误连在一起。下面只给模型一个查询付款记录的工具，并要求它提交订单号；后续两条执行路径都会使用同一份工具定义。

In [2]:
tools = [{
    "type": "function",
    "function": {
        "name": "get_payment",
        "description": "查询订单的付款记录",
        "parameters": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"}
            },
            "required": ["order_id"],
        },
    },
}]

# 工具名称会出现在后续 trace 的调用签名中
# 必填字段让每次模型决定都能关联到同一订单
print("工具名称：", tools[0]["function"]["name"])
print("必填字段：", tools[0]["function"]["parameters"]["required"])

工具名称： get_payment
必填字段： ['order_id']


输出显示模型只能调用 `get_payment`，并且必须提供 `order_id`。工具定义只统一模型决策的格式，不负责解释返回结果；下一步固定这次运行要处理的订单和消息。

## 2.2 固定任务与模型消息
这次任务只要求查询订单 `ORD-2048` 的付款状态。消息中不提前告诉模型故障原因，真实 API 只负责给出第一步工具决定；后面的 Reliability Ops 证据来自这次决定及其实际执行轨迹。

In [3]:
task = {
    "order_id": "ORD-2048",
    "request": "查询付款记录，并说明是否可以继续处理退款。",
}

# system 消息只规定模型提交工具调用
# user 消息提供本次真实 API 唯一需要读取的任务事实
messages = [
    {"role": "system", "content": "请调用 get_payment 查询付款记录。"},
    {"role": "user", "content": f"订单号：{task['order_id']}；请求：{task['request']}"},
]

print("任务：", task)
print("消息条数：", len(messages))

任务： {'order_id': 'ORD-2048', 'request': '查询付款记录，并说明是否可以继续处理退款。'}
消息条数： 2


输出固定了订单号和两条消息。基线与改进版本都会从这组输入开始；下一步准备一个生产中常见的接口版本不一致结果。

## 2.3 准备工具返回结果
生产系统经常出现工具已经返回数据，但字段名称在接口升级后发生变化。这里的返回值保留了金额，却使用 `paid` 文本而不是旧解析器期待的 `amount_cents`；这个事实会让后续重复调用有明确的共同原因。

In [4]:
payment_result = {
    "order_id": task["order_id"],
    "paid": "CNY 128.00",
}

# 返回结果确实包含订单和金额信息
# paid 的字段形状与旧解析器约定不同，构成唯一故障事实
print("工具返回：", payment_result)

工具返回： {'order_id': 'ORD-2048', 'paid': 'CNY 128.00'}


输出显示工具并非没有返回数据，问题是返回结构与旧解析约定不一致。后续实验只围绕这一个故障事实展开，不改变模型请求或工具结果；下一步固定 Reliability Ops 必须找出的诊断答案。

## 2.4 固定诊断标准
Reliability Ops 的目标不是把失败伪装成成功，而是把失败解释清楚。后续结果只有同时找出工具层的字段不一致、识别三次相同付款查询，并保留可追溯的根因，才算诊断完成。

In [5]:
expected_diagnosis = {
    "failure_module": "Tool",
    "failure_mode": "payment_result_schema_mismatch",
    "repeated_call_count": 3,
    "root_cause": "get_payment returned paid instead of amount_cents",
}

# 这些字段是基线和改进版本共同使用的唯一评分标准
# 业务任务仍然标记为失败，避免把诊断成功误写成业务成功
print(expected_diagnosis)

{'failure_module': 'Tool', 'failure_mode': 'payment_result_schema_mismatch', 'repeated_call_count': 3, 'root_cause': 'get_payment returned paid instead of amount_cents'}


输出给出了唯一诊断标准：工具层、字段不一致、三次重复调用和最早根因都必须保留。至此，工具、任务、消息、故障事实与评分标准已经固定；下一章将向真实 API 请求第一步工具决定，并记录 provider 返回的运行信息。

# 3. 获取并验证 API 响应
## 3.1 获取真实响应
第 2 章已经固定了模型消息和唯一工具。下面把它们发送给 `.env` 指定的真实模型，要求模型提交工具调用，并记录从发出请求到收到响应的实际等待时间。

In [6]:
from time import perf_counter

# 计时只覆盖本次真实 API 请求
# 原始 response 将由后续两条执行路径共同复用
request_started = perf_counter()
response = client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=tools,
    tool_choice="required",
    temperature=0,
)
api_latency_ms = round((perf_counter() - request_started) * 1000)
print("真实 API 响应已收到")

真实 API 响应已收到


输出说明真实模型已经返回，但付款工具尚未执行。完整响应保存在 `response` 中；下一步只读取模型提交的工具名称、调用编号和参数。

## 3.2 读取模型决定
模型决定保存在结构化工具请求中，而不是普通文字里。下面取出第一条工具请求，并把 JSON 参数还原成字典，后续 trace 会直接引用这份真实决定。

In [7]:
import json

# tool_call 保留 provider 分配的真实调用编号
# json.loads 把原始参数文本还原成程序可读的字典
assistant_message = response.choices[0].message
tool_call = assistant_message.tool_calls[0]
tool_arguments = json.loads(tool_call.function.arguments)

print("调用编号：", tool_call.id)
print("工具名称：", tool_call.function.name)
print("工具参数：", tool_arguments)

调用编号： call_dd6821b65bf54079a0c54771
工具名称： get_payment
工具参数： {'order_id': 'ORD-2048'}


输出展示了真实模型返回的调用编号、工具名称和订单参数。调用编号由 provider 生成，每次运行可能不同；工具名称和订单号则应与第 2 章固定的任务一致。下一步直接确认这两个任务事实。

## 3.3 确认模型决定
后续实验必须建立在模型已经选对工具和订单的前提上，否则无法判断问题究竟来自模型还是外层 Harness。下面只比较工具名称和订单号，不执行工具。

In [8]:
# 工具名称必须是第 2 章唯一提供的 get_payment
# 参数中的订单号必须与用户任务完全相同
tool_is_correct = tool_call.function.name == "get_payment"
order_is_correct = tool_arguments["order_id"] == task["order_id"]
model_decision_correct = tool_is_correct and order_is_correct

print("工具正确：", tool_is_correct)
print("订单正确：", order_is_correct)
print("模型决定正确：", model_decision_correct)

工具正确： True
订单正确： True
模型决定正确： True


三个结果都为 `True`，说明真实模型正确选择了付款查询工具和目标订单。后续发生的字段解析失败与重复调用因此属于外层执行问题；下一步保存这次请求的 provider 运行信息。

## 3.4 保存请求信息
一次工具决定还需要关联它来自哪个 provider 和模型、消耗多少 Token、等待多久以及为什么停止。下面直接读取真实响应中的原始 usage 和停止原因；provider 没有返回金额，因此成本不作估算。

In [9]:
# Token 数直接来自 provider usage，不使用字符数估算
# stop_reason 表示模型正在等待外层程序执行工具
usage = response.usage
api_metrics = {
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "input_tokens": usage.prompt_tokens,
    "output_tokens": usage.completion_tokens,
    "total_tokens": usage.total_tokens,
    "cost_usd": None,
    "latency_ms": api_latency_ms,
    "stop_reason": response.choices[0].finish_reason,
}

print(api_metrics)

{'provider': 'openai', 'model': 'LongCat-2.0', 'input_tokens': 171, 'output_tokens': 110, 'total_tokens': 281, 'cost_usd': None, 'latency_ms': 3794, 'stop_reason': 'tool_calls'}


输出保存了本次真实 API 调用的 provider、模型、Token、成本状态、延迟和停止原因。`tool_calls` 只表示模型已经提交工具请求，不表示付款查询或完整任务成功；下一章将定义只保留末次错误的生产基线组件。

# 4. 定义基线组件
## 只保留最后一条错误
生产系统中最常见的低信息基线，是任务失败后只把最后一条错误写入日志。它实现简单，却会丢掉前面的调用顺序、工具参数和错误如何传到最后一步；下面原样定义这个基线组件，暂时不运行它。

In [10]:
def keep_last_error(final_error):
    # 基线只保存最终状态和最后一条错误文字
    # 它不保存之前的步骤、工具参数或因果关系
    return {
        "status": final_error["status"],
        "message": final_error["message"],
    }

print("基线组件已定义：只保留最后一条错误")

基线组件已定义：只保留最后一条错误


输出说明基线组件已经准备好，但还没有处理第 5 章即将生成的失败轨迹。下一章会把同一份真实模型决定交给这个组件，观察它为什么只能告诉我们“失败了”，却回答不了“最早错在哪里”。

# 5. 展示基线故障
## 5.1 运行三次相同调用
旧解析器仍然读取 `amount_cents`，而第 2 章的工具结果只提供 `paid`。生产中的错误基线不会先修正协议，而是用相同参数再次调用同一工具；下面显式运行三次，观察每次得到的结果。

In [11]:
baseline_attempts = []

# 三次尝试都复用真实模型返回的同一工具名称和参数
# 旧解析器只读取 amount_cents，因此每次结果都是 None
for attempt_number in range(1, 4):
    parsed_amount = payment_result.get("amount_cents")
    attempt = {
        "attempt": attempt_number,
        "tool": tool_call.function.name,
        "arguments": tool_arguments,
        "parsed_amount": parsed_amount,
    }
    baseline_attempts.append(attempt)
    print("[中间决定]", attempt)

print("[输入上下文] 订单：", task["order_id"])

[中间决定] {'attempt': 1, 'tool': 'get_payment', 'arguments': {'order_id': 'ORD-2048'}, 'parsed_amount': None}
[中间决定] {'attempt': 2, 'tool': 'get_payment', 'arguments': {'order_id': 'ORD-2048'}, 'parsed_amount': None}
[中间决定] {'attempt': 3, 'tool': 'get_payment', 'arguments': {'order_id': 'ORD-2048'}, 'parsed_amount': None}
[输入上下文] 订单： ORD-2048


输出显示三次调用的工具、订单参数和空解析结果完全相同。重复执行没有产生新信息，只增加了调用次数；但这些过程记录尚未进入基线的最终日志，下一步只保存最后一条错误。

## 5.2 保存基线结果
三次尝试之后，任务只能以失败结束。下面把最终错误交给第 4 章的 `keep_last_error`；这个组件不会接收刚才的尝试列表，因此输出中只剩下笼统的失败状态和文字。

In [12]:
final_error = {
    "status": "failed",
    "message": "task_failed",
}

# 基线组件只接收最终错误，不接收三次尝试过程
# baseline_record 就是生产日志最终留下的全部信息
baseline_record = keep_last_error(final_error)
print("[最终行为]", baseline_record)

[最终行为] {'status': 'failed', 'message': 'task_failed'}


输出只说明任务失败，没有工具返回结构、重复次数或最早错误。此时即使三次失败刚刚发生，离开当前进程的人也无法从这条记录重建过程；下一步用统一诊断标准查看具体缺了什么。

## 5.3 查看诊断缺口
第 2 章已经固定了四项正确诊断。基线记录没有这些字段，所以下面把它能够回答的内容明确写成 `None`，再与同一份标准比较。

In [13]:
baseline_diagnosis = {
    "failure_module": None,
    "failure_mode": None,
    "repeated_call_count": None,
    "root_cause": None,
}

# 基线与改进版本始终使用第 2 章的同一份标准
# 四项信息全部缺失，因此完整诊断不能通过
baseline_passed = baseline_diagnosis == expected_diagnosis
print("基线诊断：", baseline_diagnosis)
print("诊断通过：", baseline_passed)

基线诊断： {'failure_module': None, 'failure_mode': None, 'repeated_call_count': None, 'root_cause': None}
诊断通过： False


输出中的四项诊断都是 `None`，`诊断通过` 为 `False`。模型已经选对工具和订单，付款工具也返回了金额信息；真正的问题是外层程序重复使用旧字段并只留下末次错误。下一章将定义结构化 Span、因果链、失败分类、循环签名和 Run Summary。

# 6. 定义改进组件
## 6.1 定义结构化 Span
平铺日志无法表达步骤之间的关系。下面用最少字段定义 Span：`trace_id` 表示同一次运行，`span_id` 表示当前步骤，`parent_id` 表示结构上的上级，`cause_id` 表示直接导致当前结果的前一步，其他事实放进 `attributes`。

In [14]:
def make_span(trace_id, span_id, parent_id, cause_id, kind, status, attributes):
    # parent_id 表达步骤属于哪段执行结构
    # cause_id 单独表达哪一步直接引发当前结果
    return {
        "trace_id": trace_id,
        "span_id": span_id,
        "parent_id": parent_id,
        "cause_id": cause_id,
        "kind": kind,
        "status": status,
        "attributes": attributes,
    }

print("结构化 Span 已定义")

结构化 Span 已定义


输出说明 Span 构造函数已经定义，但尚未生成任何轨迹。`parent_id` 用来重建执行结构，`cause_id` 用来回答错误从哪里传来；下一步定义如何沿因果链找到最早错误。

## 6.2 定义根因回溯
最终失败通常只是传播链的最后一站。下面先按 `span_id` 建立索引，再从最终 Span 沿 `cause_id` 一步步向前走，直到找到没有更早原因的 Span。

In [15]:
def find_root_cause(spans, final_span_id):
    # 索引让程序可以通过 span_id 直接找到对应步骤
    # cause_id 为 None 的步骤就是这条传播链的最早根因
    span_by_id = {}
    for span in spans:
        span_by_id[span["span_id"]] = span

    current_span = span_by_id[final_span_id]
    while current_span["cause_id"] is not None:
        current_span = span_by_id[current_span["cause_id"]]

    return current_span

print("根因回溯函数已定义")

根因回溯函数已定义


输出说明根因回溯函数已经准备好。它不根据错误文字猜测原因，只读取显式 `cause_id`；下一步把找到的根因按 Harness 模块归类。

## 6.3 定义失败分类
知道最早错误还不够，运维人员还要知道应该由哪个 Harness 模块处理。下面把模型、工具和任务运行器三种步骤分别映射到 Execution、Tool 和 Lifecycle，并保留 Span 中已经记录的具体失败模式。

In [16]:
def classify_failure(root_span):
    # kind 决定故障属于哪个 ETCLOVG 模块
    # failure_mode 保留该模块内部更具体的失败类型
    module_by_kind = {
        "model": "Execution",
        "tool": "Tool",
        "runner": "Lifecycle",
    }
    return {
        "failure_module": module_by_kind[root_span["kind"]],
        "failure_mode": root_span["attributes"]["failure_mode"],
    }

print("失败分类函数已定义")

失败分类函数已定义


输出说明分类函数已经定义。它会把工具步骤的失败归到 `Tool`，而不是笼统归咎于模型；下一步定义如何识别参数完全相同的重复工具调用。

## 6.4 定义重复调用统计
工具参数字典的字段顺序可能不同，但含义仍然相同。下面先把参数转成字段顺序固定的 JSON，再与工具名称拼成调用签名；相同签名出现多少次，就表示同一个动作被重复了多少次。

In [17]:
def count_repeated_calls(spans):
    # sort_keys 固定参数字段顺序，让相同调用得到相同签名
    # call_counts 按签名累计每种工具调用出现的次数
    call_counts = {}
    for span in spans:
        if span["kind"] == "tool":
            tool_name = span["attributes"]["tool_name"]
            arguments = span["attributes"]["arguments"]
            arguments_text = json.dumps(arguments, ensure_ascii=False, sort_keys=True)
            signature = tool_name + " | " + arguments_text
            call_counts[signature] = call_counts.get(signature, 0) + 1

    return max(call_counts.values())

print("重复调用统计函数已定义")

重复调用统计函数已定义


输出说明重复调用统计函数已经定义。它比较的是结构化工具名和参数，不依赖自然语言日志是否写成相同句子；最后一步把根因、分类、重复次数和真实 API 指标汇总起来。

## 6.5 定义 Run Summary
诊断结果需要一个稳定出口，供人查看或交给下游系统。下面把四项核心诊断放进 `diagnosis`，同时明确业务仍然失败，并原样关联第 3 章的真实 API 指标和 Harness 停止原因。

In [18]:
def build_run_summary(root_span, failure_label, repeated_call_count, provider_metrics):
    # diagnosis 使用第 2 章固定的四项字段
    # api_metrics 原样保留本次真实请求的 Token、成本和延迟
    diagnosis = {
        "failure_module": failure_label["failure_module"],
        "failure_mode": failure_label["failure_mode"],
        "repeated_call_count": repeated_call_count,
        "root_cause": root_span["attributes"]["exception.message"],
    }
    return {
        "business_success": False,
        "diagnosis": diagnosis,
        "api_metrics": provider_metrics,
        "harness_stop_reason": "max_attempts",
    }

print("Run Summary 函数已定义")

Run Summary 函数已定义


输出说明五个改进组件已经全部定义，但尚未处理失败轨迹。它们的职责彼此分开：Span 保存事实，因果链找根因，分类器确定模块，调用签名统计循环，Run Summary 汇总结果；下一章将用同一份真实模型决定运行完整修复路径。

# 7. 展示修复结果
## 7.1 构建结构化失败轨迹
下面复用第 3 章真实模型提交的工具名称和订单参数，记录一次模型决定、三次付款查询失败和一次最终任务失败。三次工具 Span 使用同一个调用签名，但通过 `cause_id` 串成错误传播链。

In [19]:
trace_id = "trace-ORD-2048"
failure_attributes = {
    "tool_name": tool_call.function.name,
    "arguments": tool_arguments,
    "failure_mode": expected_diagnosis["failure_mode"],
    "exception.type": "SchemaMismatch",
    "exception.message": expected_diagnosis["root_cause"],
}

# model Span 保存真实 API 的正确工具决定
# tool 和 runner Span 保存失败状态及其直接因果关系
trace_spans = [
    make_span(trace_id, "span-model", None, None, "model", "ok", {
        "tool_name": tool_call.function.name,
        "arguments": tool_arguments,
        "model": model_name,
    }),
    make_span(trace_id, "span-tool-1", "span-model", None, "tool", "error", failure_attributes),
    make_span(trace_id, "span-tool-2", "span-model", "span-tool-1", "tool", "error", failure_attributes),
    make_span(trace_id, "span-tool-3", "span-model", "span-tool-2", "tool", "error", failure_attributes),
    make_span(trace_id, "span-runner", "span-model", "span-tool-3", "runner", "error", {
        "message": "task_failed",
    }),
]

print("[输入上下文] 订单：", task["order_id"], "trace_id：", trace_id)
for span in trace_spans:
    print("[中间决定]", span["span_id"], span["kind"], span["status"], "cause=", span["cause_id"])
print("[最终产物] Span 数量：", len(trace_spans))

[输入上下文] 订单： ORD-2048 trace_id： trace-ORD-2048
[中间决定] span-model model ok cause= None
[中间决定] span-tool-1 tool error cause= None
[中间决定] span-tool-2 tool error cause= span-tool-1
[中间决定] span-tool-3 tool error cause= span-tool-2
[中间决定] span-runner runner error cause= span-tool-3
[最终产物] Span 数量： 5


输出显示五个 Span 已经连接起来：模型决定本身成功，三次工具失败沿 `cause_id` 传递，最后由 runner 报告任务失败。下一步从最终 Span 反向追到最早的工具错误。

## 7.2 回溯最早根因
最终的 `span-runner` 只说明任务停了，真正的根因在更早的工具 Span。下面调用第 6 章的回溯函数，沿着 `span-runner → span-tool-3 → span-tool-2 → span-tool-1` 找到链条起点。

In [20]:
# 从最终失败 Span 开始，而不是从日志文字猜测根因
# root_span 将保存最早出现的结构不一致事实
root_span = find_root_cause(trace_spans, "span-runner")

print("根因 Span：", root_span["span_id"])
print("根因信息：", root_span["attributes"]["exception.message"])

根因 Span： span-tool-1
根因信息： get_payment returned paid instead of amount_cents


输出把根因定位到 `span-tool-1`，而不是最后的 runner。根因文字明确说明旧解析器期待 `amount_cents`，工具实际返回了 `paid`；下一步把这个 Span 归类到具体 Harness 模块。

## 7.3 归类失败模块
根因已经找到，但还需要决定由哪一层负责修复。下面只把根因 Span 交给分类器，保留模块名称和具体失败模式。

In [21]:
# 分类器只读取根因 Span 的 kind 和 failure_mode
# 这里的 kind=tool 应归入 Tool 层，而不是归咎于模型
failure_label = classify_failure(root_span)
print("失败分类：", failure_label)

失败分类： {'failure_module': 'Tool', 'failure_mode': 'payment_result_schema_mismatch'}


输出把失败归类为 `Tool`，模式为 `payment_result_schema_mismatch`。这说明模型决策正确，错误发生在工具结果与 Harness 解析约定之间；下一步统计相同工具调用实际重复了几次。

## 7.4 统计重复调用
三次工具 Span 的文字描述可以不同，但工具名和参数完全相同。下面用第 6 章的规范化签名统计同一动作出现的次数，不依赖日志措辞。

In [22]:
# 统计只读取结构化 tool Span，不把 model 或 runner 算作工具调用
# repeated_call_count 将直接进入最终 Run Summary
repeated_call_count = count_repeated_calls(trace_spans)
print("重复调用次数：", repeated_call_count)

重复调用次数： 3


输出显示同一个 `get_payment(ORD-2048)` 调用了 3 次，正好还原基线的重复行为。现在根因、模块和循环证据都已经准备好；下一步把它们与真实 API 指标合成一份运行摘要。

## 7.5 生成 Run Summary
一个可靠性组件最终要给出稳定、可交接的结果。下面调用 Run Summary 构造函数，明确业务仍然失败，但诊断已经完整，并保留第 3 章真实请求的 provider、Token、成本和延迟。

In [23]:
# summary 同时保存业务状态、诊断结果和真实 API 运行指标
# 不把诊断成功误写成业务任务成功
run_summary = build_run_summary(
    root_span,
    failure_label,
    repeated_call_count,
    api_metrics,
)

print(json.dumps(run_summary, ensure_ascii=False, indent=2))

{
  "business_success": false,
  "diagnosis": {
    "failure_module": "Tool",
    "failure_mode": "payment_result_schema_mismatch",
    "repeated_call_count": 3,
    "root_cause": "get_payment returned paid instead of amount_cents"
  },
  "api_metrics": {
    "provider": "openai",
    "model": "LongCat-2.0",
    "input_tokens": 171,
    "output_tokens": 110,
    "total_tokens": 281,
    "cost_usd": null,
    "latency_ms": 3794,
    "stop_reason": "tool_calls"
  },
  "harness_stop_reason": "max_attempts"
}


输出把业务失败与诊断成功分开：`business_success` 仍为 `False`，但 diagnosis 已包含工具层、字段不一致、3 次重复和最早根因，同时保留真实 API 指标。最后用第 2 章的唯一标准确认修复方向是否完整。

## 7.6 判断诊断结果
改进版的目标是解释失败，不是掩盖失败。下面只比较 Run Summary 中的四项 diagnosis 与共同标准，确认结构化路径是否找回了基线丢失的信息。

In [24]:
# diagnosis 必须与第 2 章的四项标准完全一致
# business_success 仍为 False，表示没有伪造业务结案
diagnostic_success = run_summary["diagnosis"] == expected_diagnosis
print("诊断结果：", run_summary["diagnosis"])
print("诊断成功：", diagnostic_success)
print("业务成功：", run_summary["business_success"])

诊断结果： {'failure_module': 'Tool', 'failure_mode': 'payment_result_schema_mismatch', 'repeated_call_count': 3, 'root_cause': 'get_payment returned paid instead of amount_cents'}
诊断成功： True
业务成功： False


输出中的 `诊断成功` 为 `True`，而 `业务成功` 仍为 `False`。改进版没有修改付款工具或伪造结案文件，却从同一条失败轨迹中恢复了根因、模块、重复次数和运行指标；下一章将把基线与改进版放在同一张消融表中比较。

# 8. 汇总消融对照
## 8.1 整理两条同源路径
基线与改进版复用同一个真实模型决定、同一份工具结果和同一次 API 指标，业务结果也都保持失败。唯一变量是外层程序是否保存结构化 Span 并执行根因、分类、重复调用和摘要处理；下面把两条路径整理成相同字段。

In [25]:
# 两条路径共享同一次真实 API 调用及其全部指标
# 对照只改变 Harness 是否保留并消费结构化失败轨迹
ablation_rows = [
    {
        "版本": "末次错误基线",
        "业务成功": False,
        "诊断成功": baseline_passed,
        "根因": baseline_diagnosis["root_cause"],
        "重复调用": baseline_diagnosis["repeated_call_count"],
        "结构化 Span": 0,
        "API 次数": 1,
        "真实 Token": api_metrics["total_tokens"],
        "成本(USD)": api_metrics["cost_usd"],
        "API 延迟(ms)": api_metrics["latency_ms"],
    },
    {
        "版本": "Reliability Ops",
        "业务成功": run_summary["business_success"],
        "诊断成功": diagnostic_success,
        "根因": run_summary["diagnosis"]["root_cause"],
        "重复调用": run_summary["diagnosis"]["repeated_call_count"],
        "结构化 Span": len(trace_spans),
        "API 次数": 1,
        "真实 Token": api_metrics["total_tokens"],
        "成本(USD)": api_metrics["cost_usd"],
        "API 延迟(ms)": api_metrics["latency_ms"],
    },
]

print("对照版本数：", len(ablation_rows))
print("共享真实 API 次数：", 1)

对照版本数： 2
共享真实 API 次数： 1


输出说明两条消融记录已经准备好，并且共同使用一次真实 API 调用。下一步按固定列打印完整对照，直接观察哪些事实保持不变、哪些诊断能力发生变化。

## 8.2 展示完整对照
下面使用纯文本表格横向展示两条路径。`None` 表示基线没有留下对应证据，成本也保持 provider 的未知状态；它们都不会被改写成零。

In [26]:
# columns 固定字段顺序，避免两行使用不同口径
# 内层循环逐列取值，完整展示每个对照字段
columns = list(ablation_rows[0])
print(" | ".join(columns))

for row in ablation_rows:
    values = []

    for column in columns:
        values.append(str(row[column]))

    print(" | ".join(values))

版本 | 业务成功 | 诊断成功 | 根因 | 重复调用 | 结构化 Span | API 次数 | 真实 Token | 成本(USD) | API 延迟(ms)
末次错误基线 | False | False | None | None | 0 | 1 | 281 | None | 3794
Reliability Ops | False | True | get_payment returned paid instead of amount_cents | 3 | 5 | 1 | 281 | None | 3794


表格显示两条路径的业务结果、API 次数、Token、成本状态和 API 延迟完全相同。基线没有结构化证据，无法给出根因和重复次数；Reliability Ops 保存 5 个 Span 后，完整诊断从 `False` 变为 `True`。

## 8.3 生成最终 Eval Report
最后把最重要的因果关系收束成一份结构化报告。只有模型和真实 API 用量保持相同、基线诊断失败且改进诊断成功时，才能说明变化来自外层 Reliability Ops。

In [27]:
# same_api_facts 确认消融两端没有更换模型请求或用量
# binding_constraint_confirmed 只在诊断结果按预期翻转时成立
same_api_facts = (
    ablation_rows[0]["API 次数"] == ablation_rows[1]["API 次数"]
    and ablation_rows[0]["真实 Token"] == ablation_rows[1]["真实 Token"]
    and ablation_rows[0]["API 延迟(ms)"] == ablation_rows[1]["API 延迟(ms)"]
)
binding_constraint_confirmed = (
    same_api_facts
    and baseline_passed is False
    and diagnostic_success is True
)
eval_report = {
    "notebook": "05_O_nanoReliabilityOps_minimal.ipynb",
    "layer": "O",
    "provider": config["NANO_BACKEND"],
    "model": model_name,
    "real_api_rollouts": 1,
    "baseline_diagnostic_success": baseline_passed,
    "reliability_ops_diagnostic_success": diagnostic_success,
    "business_success": run_summary["business_success"],
    "same_api_facts": same_api_facts,
    "binding_constraint_confirmed": binding_constraint_confirmed,
    "artifact": run_summary,
}

print(json.dumps(eval_report, ensure_ascii=False, indent=2))

{
  "notebook": "05_O_nanoReliabilityOps_minimal.ipynb",
  "layer": "O",
  "provider": "openai",
  "model": "LongCat-2.0",
  "real_api_rollouts": 1,
  "baseline_diagnostic_success": false,
  "reliability_ops_diagnostic_success": true,
  "business_success": false,
  "same_api_facts": true,
  "binding_constraint_confirmed": true,
  "artifact": {
    "business_success": false,
    "diagnosis": {
      "failure_module": "Tool",
      "failure_mode": "payment_result_schema_mismatch",
      "repeated_call_count": 3,
      "root_cause": "get_payment returned paid instead of amount_cents"
    },
    "api_metrics": {
      "provider": "openai",
      "model": "LongCat-2.0",
      "input_tokens": 171,
      "output_tokens": 110,
      "total_tokens": 281,
      "cost_usd": null,
      "latency_ms": 3794,
      "stop_reason": "tool_calls"
    },
    "harness_stop_reason": "max_attempts"
  }
}


Eval Report 中 `same_api_facts` 和 `binding_constraint_confirmed` 都为 `True`：模型与真实 API 事实没有变化，诊断却从失败变为成功。这直接说明本任务的可靠性瓶颈在外层 Harness 是否保留并消费结构化故障证据。

## 8.4 拓展

### nano 版省略了什么

nano 版只诊断一条短 trace，没有在线告警、SLO、采样、去重、关联分析、异常检测、自动缓解、事故时间线和跨服务根因定位。生产 Reliability Ops 还需把模型随机性、基础设施噪声和 Harness 缺陷分开，并让每个修复建议都能回溯到具体证据。

### 延伸阅读

1. 2026, [Agentic Harness Engineering](https://arxiv.org/abs/2604.25850)：以可观测信号驱动 Coding-Agent Harness 的自动诊断与演化。
2. 2026, [Anthropic, Quantifying infrastructure noise in agentic coding evals](https://www.anthropic.com/engineering/infrastructure-noise)：识别环境噪声对可靠性结论的污染。
3. 2026, [OpenTelemetry, GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/)：跨模型、工具和 Agent 步骤的统一遥测字段。